# Transfer Resolution Models

Previous focus structure effects (see notebooks 01 to 06) as well as furhter exhaustive modelling (see notebooks 17_appendix vs. 18_appendix) show consistently the heterogeneity of combined data; by spliting data into goal contexts and no-goal contexts, both effects as well as models improved despite the pruning of observations. No-goal contexts are also analysed separately (notebook 11); however, and although results are coherent with what is seen by comparing combined data to goal behaviour data, the sample is clearly not enough to show anything significant from no-goal contexts.

Taking into account the effects summary (notebook 05) and response opportunity analysis (notebook 06), these were chosen to be the best models to predict participants resolving by means of transfer. Predictors were added only when supported by the opportunity analyses.

Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Goal Behaviour Subgroup

1. Create resolved_transfer 
2. Counts and Proportions
3. Logistic Models

    Main Effect Model:

    - RT01: Focus

    Additive Model:

    - RT05: Focus + Goal_Type
    - RT07: Focus + Agent + Goal_Type

    Interactive Model:

    - RT11: Focus * Agent + Goal_Type
4. Summary

Read Data

In [2]:
subgroup_theoretical = pd.read_csv("../../data/processed/subgroup_theoretical.csv")

### 1. Create resolved_transfer

Binary Outcome: transfer responses from those observations that managed to escape L2_other options.

In [3]:
subgroup_theoretical["escape_L2"] = (
    subgroup_theoretical["Response_Full"] != "L2_other"
).astype(int)

In [4]:
escapees = subgroup_theoretical[subgroup_theoretical["escape_L2"] == 1].copy()

In [5]:
escapees.shape

(160, 33)

In [6]:
escapees["resolved_transfer"] = (escapees["Response_Full"] == "L1_transfer").astype(int)

Gather goal-context data

In [7]:
goals_escapees = escapees[escapees["Goal_Type"] != "no_goal"].copy()

In [8]:
goals_escapees.shape

(106, 34)

### 2. Counts and Proportions

resolved_transfer responses count for goals:

In [9]:
goals_resolved_transfer_counts = pd.crosstab(
    goals_escapees["resolved_transfer"],
    goals_escapees["Response_Full"],
    margins=True)

goals_resolved_transfer_counts

Response_Full,L1_transfer,correct,missing_response,All
resolved_transfer,,,,
0,0,43,1,44
1,62,0,0,62
All,62,43,1,106


resolved_transfer responses proportions for goals:

In [10]:
goals_resolved_transfer_props = goals_escapees["resolved_transfer"].value_counts(normalize=True)
goals_resolved_transfer_props

resolved_transfer
1    0.584906
0    0.415094
Name: proportion, dtype: float64

From 106 observations, 58.4% of goal responses that had escaped the L2-other landscape were resolved by means of transfer.

Counts and Proportions by Condition:

In [11]:
goals_counts_condition = pd.crosstab(
    [goals_escapees["Goal_Type"], goals_escapees["Agent"], goals_escapees["Focus"]],
    goals_escapees["resolved_transfer"],
    margins = True)

goals_counts_condition

resolved_transfer               0   1  All
Goal_Type         Agent Focus             
goal_frequent     0     I      10   6   16
                        They    5   7   12
                  1     I       7   7   14
                        They    3   5    8
goal_non_frequent 0     I       9   7   16
                        They    3  14   17
                  1     I       4  11   15
                        They    3   5    8
All                            44  62  106

In [12]:
goals_props_condition = pd.crosstab(
    [goals_escapees["Goal_Type"], goals_escapees["Agent"], goals_escapees["Focus"]],
    goals_escapees["resolved_transfer"],
    normalize= "index")

goals_props_condition

resolved_transfer                     0         1
Goal_Type         Agent Focus                    
goal_frequent     0     I      0.625000  0.375000
                        They   0.416667  0.583333
                  1     I      0.500000  0.500000
                        They   0.375000  0.625000
goal_non_frequent 0     I      0.562500  0.437500
                        They   0.176471  0.823529
                  1     I      0.266667  0.733333
                        They   0.375000  0.625000

### 3. Logistic Models

Import Libraries

In [13]:
import statsmodels.formula.api as smf
from scipy.stats import chi2

Sanity check:

In [14]:
goals_escapees["resolved_transfer"].value_counts()

resolved_transfer
1    62
0    44
Name: count, dtype: int64

**Main Effect Model**

**RT01**

resolved_transfer ~ Focus

In [15]:
RT01 = smf.logit(
    "resolved_transfer ~ Focus",
    data=goals_escapees
    ).fit()

print(RT01.summary())

Optimization terminated successfully.
         Current function value: 0.662011
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:      resolved_transfer   No. Observations:                  106
Model:                          Logit   Df Residuals:                      104
Method:                           MLE   Df Model:                            1
Date:                Mon, 21 Sep 2026   Pseudo R-squ.:                 0.02453
Time:                        13:56:09   Log-Likelihood:                -70.173
converged:                       True   LL-Null:                       -71.938
Covariance Type:            nonrobust   LLR p-value:                   0.06029
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         0.0328      0.256      0.128      0.898      -0.469       0.535
Focus[T.They]     0.

The effect of Focus on its own seems to be quite defined but not significant yet. 

**Additive Models**

**RT05**

resolved_transfer ~ Focus + Goal_Type

In [16]:
RT05 = smf.logit(
    "resolved_transfer ~ Focus + Goal_Type",
    data=goals_escapees
    ).fit()

print(RT05.summary())

Optimization terminated successfully.
         Current function value: 0.649658
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:      resolved_transfer   No. Observations:                  106
Model:                          Logit   Df Residuals:                      103
Method:                           MLE   Df Model:                            2
Date:                Mon, 21 Sep 2026   Pseudo R-squ.:                 0.04273
Time:                        13:56:09   Log-Likelihood:                -68.864
converged:                       True   LL-Null:                       -71.938
Covariance Type:            nonrobust   LLR p-value:                   0.04623
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -0.2983      0.331     -0.901     

Even though the coefficients do not make it to significance, the model is significant overall. This is interesting as it does give way to consider that, despite not having enough power, Goal Type could after all play a role as expected from the effects and opportunity analyses.

**RT07**

resolved_transfer ~ Focus + Agent + Goal_Type

In [17]:
RT07= smf.logit(
    "resolved_transfer ~ Focus + Agent + Goal_Type",
    data=goals_escapees
    ).fit()

print(RT07.summary())

Optimization terminated successfully.
         Current function value: 0.645294
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:      resolved_transfer   No. Observations:                  106
Model:                          Logit   Df Residuals:                      102
Method:                           MLE   Df Model:                            3
Date:                Mon, 21 Sep 2026   Pseudo R-squ.:                 0.04916
Time:                        13:56:09   Log-Likelihood:                -68.401
converged:                       True   LL-Null:                       -71.938
Covariance Type:            nonrobust   LLR p-value:                   0.06960
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -0.4952      0.393     -1.262     

The full additive model, unlike for escaping the L2, is not significant. Yet, both the Focus as well as the goal type coefficients have improved once agency is in the model. Howewer, as expected from previous analyses, the relationship focus-agency is more likely interactive. 

**Interactive Model**

**RT11**

resolved_transfer ~ Focus * Agent + Goal_Type

In [18]:
RT11 = smf.logit(
    "resolved_transfer ~ Focus * Agent + Goal_Type",
    data=goals_escapees
    ).fit()

print(RT11.summary())

Optimization terminated successfully.
         Current function value: 0.634611
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:      resolved_transfer   No. Observations:                  106
Model:                          Logit   Df Residuals:                      101
Method:                           MLE   Df Model:                            4
Date:                Mon, 21 Sep 2026   Pseudo R-squ.:                 0.06490
Time:                        13:56:09   Log-Likelihood:                -67.269
converged:                       True   LL-Null:                       -71.938
Covariance Type:            nonrobust   LLR p-value:                   0.05318
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -0.7155      0.425     -1.682     

- The Focus' coefficient is significant in the interactive model. The Agent's coefficient, while not significant, has improved a lot compared to the full additive model confirming that the relationship is interactive even if the interaction's coefficient is not. 

- Goal type's contribution remains very steady, with a very similar contribution (coeff = 0.6518) compared to the additive model Focus-Goal_Type RT05 (coeff = 0.6531) and p-value = 0.115 (vs. 0.108). 

### 4. Summary 

- Focus emerged as the primary determinant of transfer resolution (RT01). Opportunity analysis suggested that Agency modified this relationship and that Goal Type contributed systematic differences. Consequently, the Focus × Agent + Goal Type model (RT11) was retained.

- External focus (Focus = They) strongly increases transfer when agency is implicit (Agent = 0) or, from opportunity analyses, external focus redirects responses towards transfer. 

- Explicit agency (Agent = 1) is associated with a higher probability of transfer when Focus = I. This explains why explicit self-agency favoured transfer in opportunity analysis and EDA. Even though the evidence is not conventionally significant, the coefficient is substantial and points in the right direction. This opens up discussion regarding how costly embodied responsability/accountability is and how removing direct responsability by using implicit agency showed better efficiency.

- The negative interaction explains why explicit agency does not have the same effect under external focus (They). 

- Non-frequent goals generally raise transfer probability or channel more responses toward transfer.

- However, from opportunity analyses, the model cannot explain the large increase in transfer observed for goal_non_frequent + Agent = 0 + They. In order to do that, we would need a full three-way interaction and we already know that the data does not have enough power to estimate that.